# GROQ Model Integration

In [7]:
import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
load_dotenv()

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = init_chat_model("groq:qwen/qwen3-32b")
response = model.invoke("Why do parrots talk?")
response

AIMessage(content='<think>\nOkay, so I need to figure out why parrots talk. Let me start by recalling what I know about parrots. I know they\'re birds that can mimic human speech. But why do they do that? Maybe it\'s related to communication. I remember that in the wild, parrots use sounds to communicate with each other, like for finding mates or warning others about danger. So maybe talking is an extension of that ability.\n\nAlso, parrots have a syrinx, which is their vocal organ. I think it\'s different from the human larynx. Does that help them mimic sounds? Maybe the structure of their syrinx allows for more flexibility in producing a variety of sounds, which is why they can imitate human speech. But how does that translate to actually learning the words?\n\nI\'ve heard that some parrots can associate words with meanings. For example, a parrot might say "hello" when someone enters the room. So maybe they\'re learning through repetition and context. They observe when their owners s

In [11]:
import os
from langchain_groq import ChatGroq

model = ChatGroq(model="qwen/qwen3-32b")
response = model.invoke("Why do parrots talk?")
response

AIMessage(content='<think>\nOkay, so I need to figure out why parrots talk. Let me start by recalling what I know about parrots. They\'re birds, and many species can mimic human speech. But why do they do that? I remember hearing something about communication. Maybe they use it to interact with other parrots? But in captivity, they talk to humans. Is it instinct or learned behavior?\n\nFirst, I should think about their natural habitat. In the wild, parrots use a variety of sounds to communicate with each other. These could include calls to signal danger, location, or to maintain contact with their flock. If they mimic human speech, it might be an extension of this vocal ability. But why would they specifically mimic human words when they\'re with humans? Maybe they learn it through interaction with people, similar to how some animals can be trained.\n\nI also heard that parrots have a vocal organ called the syrinx, which allows them to produce a wide range of sounds. This might be why 

# Streaming & Batch

## Streaming

Most model can stream their output content while it is being generated. By displaying output progressively, streaming significancy improves user experience, particularly for longer responses. Calling stream() returns an iterator that yields output chunks as the are produced. You can use a lop to process each chunk in real time.

In [1]:
from langchain.agents import create_agent


def get_weather(city: str) -> str:
    """Get weather for a given city."""

    return f"It's always sunny in {city}!"

agent = create_agent(
    model="gpt-5-nano",
    tools=[get_weather],
)
for chunk in agent.stream(
    {"messages": [{"role": "user", "content": "What is the weather in SF?"}]},
    stream_mode="updates",
    version="v2",
):
    if chunk["type"] == "updates":
        for step, data in chunk["data"].items():
            print(f"step: {step}")
            print(f"content: {data['messages'][-1].content_blocks}")

step: model
content: [{'type': 'tool_call', 'name': 'get_weather', 'args': {'city': 'San Francisco'}, 'id': 'call_C8rCNxTSfduxdYnrKZDOrcpw'}]
step: tools
content: [{'type': 'text', 'text': "It's always sunny in San Francisco!"}]
step: model
content: [{'type': 'text', 'text': "It's always sunny in San Francisco! If you'd like real-time details (temperature, wind, humidity, hourly forecast), I can pull those up—want me to fetch the latest?"}]


In [10]:
model.invoke("Write me a 200 words paragraph on Artificial Intelligence.")

AIMessage(content='<think>\nOkay, the user wants a 200-word paragraph on Artificial Intelligence. Let me start by defining AI and then mention its applications. I should include examples like healthcare, finance, and autonomous vehicles. Maybe touch on machine learning and natural language processing as subfields. Also, mention the benefits but address the ethical concerns like bias and job displacement. Need to balance the positives with the challenges. Keep the tone informative but not too technical. Let me check the word count as I go. Wait, is there a specific aspect they want to focus on? The query is pretty general, so a broad overview is safe. Make sure to conclude with the future implications and the importance of responsible development. Avoid jargon, keep it clear. Let me structure it: intro, applications, subfields, benefits, challenges, future. That should cover all bases. Alright, start drafting each section concisely. Need to stay within 200 words, so be concise. Let me c

In [12]:
for chunk in model.stream(
    "Write me a 200 words paragraph on Artificial Intelligence."
):
    print(chunk.text, end="|", flush=True)

|<think>|
|Okay|,| the| user| wants| a| |2|0|0|-word| paragraph| on| Artificial| Intelligence|.| Let| me| start| by| defining| AI| clearly|.| I| should| mention| its| key| areas| like| machine| learning| and| deep| learning|.| Maybe| include| examples| like| image| recognition| or| language| processing| to| make| it| rel|atable|.

|Wait|,| the| user| might| be| a| student| needing| an| overview| for| a| school| project|.| They| might| also| want| to| know| applications| in| different| fields|.| I| should| cover| healthcare|,| finance|,| and| transportation|.| Also|,| touch| on| the| ethical| concerns| to| show| a| balanced| view|.

|Hmm|,| should| I| mention| specific| technologies| like| neural| networks|?| Yes|,| but| keep| it| simple|.| Avoid| j|argon|.| Make| sure| to| highlight| both| benefits| and| challenges|.| Maybe| end| with| the| future| potential| and| the| need| for| responsible| development|.

|Check| the| word| count|.| Let| me| structure| the| paragraph|:| start| with| 

## Batch

Batching a collection of independent request to a model can significalle improve performance and reduce costs, as the processing can be donde in parallel

In [13]:
responses = model.batch([
    "Why is the sky blue?",
    "What is the capital of France?",
    "What is the meaning of life?",
])

for response in responses:
    print(response)

content="<think>\nOkay, so I need to figure out why the sky is blue. Let me start by recalling what I know. I remember from school that it has something to do with the sun's light and the atmosphere. But I'm a bit fuzzy on the details.\n\nFirst, I know that sunlight is white, but it's made up of all the colors of the rainbow—red, orange, yellow, green, blue, indigo, violet. Each color has a different wavelength. I think blue has a shorter wavelength, and red is longer. Maybe the different colors interact with the atmosphere differently?\n\nThe atmosphere is made up of gases like nitrogen and oxygen, right? And other particles. When sunlight enters the atmosphere, it collides with these molecules. I think there's a term for this scattering of light. Rayleigh scattering? Yeah, I think that's the one. So Rayleigh scattering has to do with the wavelength of the light. Shorter wavelengths scatter more than longer ones. So blue light scatters more than red. That's why the sky is blue. But wa